SETUP

In [2]:
# If Colab asks to restart after installing, accept it.
!pip -q install --upgrade pip
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.0" "sentence-transformers>=3.0.1" \
                 "faiss-cpu>=1.8.0" "langchain-text-splitters>=0.3.0" "pypdf>=4.2.0" \
                 "gradio>=4.44.0"
# bitsandbytes is optional (for 8-bit loading if a GPU is available). It may fail on CPU-only.
!pip -q install bitsandbytes==0.44.1 || echo "bitsandbytes optional install skipped"
!pip -q install -U google-generativeai
!pip install pypdf python-docx

ERROR: Could not find a version that satisfies the requirement bitsandbytes==0.44.1 (from versions: 0.31.8, 0.32.0, 0.32.1, 0.32.2, 0.32.3, 0.33.0, 0.33.1, 0.34.0, 0.35.0, 0.35.1, 0.35.2, 0.35.3, 0.35.4, 0.36.0, 0.36.0.post1, 0.36.0.post2, 0.37.0, 0.37.1, 0.37.2, 0.38.0, 0.38.0.post1, 0.38.0.post2, 0.38.1, 0.39.0, 0.39.1, 0.40.0, 0.40.0.post1, 0.40.0.post2, 0.40.0.post3, 0.40.0.post4, 0.40.1, 0.40.1.post1, 0.40.2, 0.41.0, 0.41.1, 0.41.2, 0.41.2.post1, 0.41.2.post2, 0.41.3, 0.41.3.post1, 0.41.3.post2, 0.42.0, 0.49.0, 0.49.1, 0.49.2)
ERROR: No matching distribution found for bitsandbytes==0.44.1
bitsandbytes optional install skipped


In [3]:
import os, torch, faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch: 2.9.0 | CUDA available: False


In [4]:
from google.colab import userdata
import os

# Retrieve secret
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Export it so the rest of the notebook sees it
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Gemini key loaded:", "✅" if GOOGLE_API_KEY else "❌ Missing")

Gemini key loaded: ✅


CONFIGURE MODEL BACKEND

In [5]:
# === Choose your generator backend ===
# "local" uses a small open-source chat model via Hugging Face Transformers.
# "openai" uses OpenAI's API (set OPENAI_API_KEY).
# "gemini" uses Google Generative AI (set GOOGLE_API_KEY).
GEN_BACKEND = "gemini"  # options: "local", "openai", "gemini"

# Local model config:
LOCAL_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # light model for CPU/GPU demos

# Retrieval config
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 4

# Prompt template
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the provided context to answer the user's question.\n"
    "If the answer is not in the context, say you don't know.\n"
)

ANSWER_TEMPLATE = """[System]
{system}

[Context]
{context}

[User Question]
{question}

[Instructions]
- Cite the most relevant chunks briefly (e.g., 'From chunk 2').
- If unsure, say 'I don't know from the provided docs.'
- Keep answers concise and factual.
"""

OPENAI_API_KEY = globals().get("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY", ""))
GOOGLE_API_KEY = globals().get("GOOGLE_API_KEY", os.getenv("GOOGLE_API_KEY", ""))

LOAD FILES/CHUNKS AND EMBEDDED

In [6]:
import os
from typing import List, Dict
from pypdf import PdfReader
from docx import Document  # ADDED THIS LINE

def load_texts_from_paths(paths: List[str]) -> List[Dict]:
    """
    Load text from various file formats: PDF, DOCX, DOC, TXT, MD
    """
    docs = []
    for p in paths:
        text = ""

        # PDF files
        if p.lower().endswith(".pdf"):
            try:
                reader = PdfReader(p)
                for page in reader.pages:
                    text += page.extract_text() or ""
                print(f"✓ Loaded PDF: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse PDF {p}: {e}")
                continue

        # DOCX files (Word documents) # NEW
        elif p.lower().endswith(".docx"):
            try:
                doc = Document(p)
                # Extract text from all paragraphs
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                # Also extract text from tables
                for table in doc.tables:
                    for row in table.rows:
                        for cell in row.cells:
                            text += "\n" + cell.text
                print(f"✓ Loaded DOCX: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOCX {p}: {e}")
                continue

        # DOC files (legacy Word format) # NEW
        elif p.lower().endswith(".doc"):
            try:
                doc = Document(p)
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                print(f"✓ Loaded DOC: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOC {p}: {e}")
                print("     Note: Legacy .doc files may require conversion to .docx")
                continue

        # TXT and MD files
        elif p.lower().endswith((".txt", ".md")):
            try:
                with open(p, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                print(f"✓ Loaded TXT/MD: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to read text file {p}: {e}")
                continue

        else:
            print(f"[SKIP] Unsupported file type: {p}")
            continue

        if text.strip():  # Only add if we got some text
            docs.append({"path": p, "text": text})
        else:
            print(f"[WARN] No text extracted from {p}")

    return docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)

def chunk_docs(docs: List[Dict]) -> List[Dict]:
    chunks = []
    for d in docs:
        for i, ch in enumerate(splitter.split_text(d["text"])):
            chunks.append({"source": d["path"], "chunk_id": i, "text": ch})
    return chunks

class RAGIndex:
    def __init__(self, embedding_model_name: str):
        self.model = SentenceTransformer(embedding_model_name, device=device)
        self.index = None
        self.chunks: List[Dict] = []

    def build(self, chunks: List[Dict]):
        self.chunks = chunks
        embs = self.model.encode([c["text"] for c in chunks], convert_to_numpy=True, show_progress_bar=True)
        dim = embs.shape[1]
        index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embs)
        index.add(embs)
        self.index = index
        print(f"Built index with {len(chunks)} chunks.")

    def search(self, query: str, k: int = 4):
        if self.index is None or not self.chunks:
            return []
        q = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q)
        scores, idxs = self.index.search(q, k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1: continue
            results.append((float(score), self.chunks[idx]))
        return results

rag = RAGIndex(EMBEDDING_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GENERATIONS BACKEND

In [7]:
def render_context(snippets):
    lines = []
    for rank, (score, ch) in enumerate(snippets, start=1):
        header = f"[Chunk {rank}] (score={score:.3f}) source={os.path.basename(ch['source'])} id={ch['chunk_id']}"
        lines.append(header + "\n" + ch["text"])
    return "\n\n".join(lines)

def build_prompt(question, context_blocks):
    return ANSWER_TEMPLATE.format(
        system=SYSTEM_PROMPT.strip(),
        context=context_blocks.strip(),
        question=question.strip()
    )

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

_local_pipe = None

def get_local_pipe():
    global _local_pipe
    if _local_pipe is None:
        tok = AutoTokenizer.from_pretrained(LOCAL_MODEL, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            low_cpu_mem_usage=True
        )
        _local_pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tok,
            device=0 if torch.cuda.is_available() else -1,
            max_new_tokens=384,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.05
        )
    return _local_pipe

def generate_local(prompt: str) -> str:
    p = get_local_pipe()
    out = p(prompt, pad_token_id=p.tokenizer.eos_token_id)[0]["generated_text"]
    return out[len(prompt):].strip()

In [9]:
def generate_openai(prompt: str) -> str:
    if not OPENAI_API_KEY:
        return "OPENAI_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":SYSTEM_PROMPT},
                      {"role":"user","content":prompt}],
            temperature=0.2,
            max_tokens=400
        )
        return r.choices[0].message.content
    except Exception as e:
        return f"[OpenAI error] {e}"

In [10]:
def generate_gemini(prompt: str) -> str:
    if not GOOGLE_API_KEY:
        return "GOOGLE_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."
    try:
        import google.generativeai as genai
        genai.configure(api_key=GOOGLE_API_KEY)
        model = genai.GenerativeModel("gemini-2.5-flash-lite")
        r = model.generate_content(prompt)
        return r.text
    except Exception as e:
        return f"[Gemini error] {e}"

ASK QUESTIONS

In [11]:
def answer_question(question: str, top_k: int = TOP_K):
    hits = rag.search(question, k=top_k)
    context = render_context(hits)
    prompt = build_prompt(question, context)

    if GEN_BACKEND == "local":
        answer = generate_local(prompt)
    elif GEN_BACKEND == "openai":
        answer = generate_openai(prompt)
    elif GEN_BACKEND == "gemini":
        answer = generate_gemini(prompt)
    else:
        answer = f"Unknown backend: {GEN_BACKEND}"

    return {"question": question, "answer": answer, "top_chunks": hits}

print("RAG ready. After indexing, call: answer_question('Your query')")

RAG ready. After indexing, call: answer_question('Your query')


In [12]:
import google.generativeai as genai, os
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

available = [m.name for m in genai.list_models()
             if "generateContent" in getattr(m, "supported_generation_methods", [])]
for name in available:
    print(name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [12]:
import google.generativeai as genai, time, os
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
m = genai.GenerativeModel("gemini-2.5-flash-lite")
t=time.time()
print(m.generate_content("Say only: ready").text, "⏱", round(time.time()-t,2), "s")

ready ⏱ 0.62 s


UPLOAD FILESAND BUILD INDEX

In [13]:
from google.colab import files

def upload_files():
    print("Select PDFs/TXT/MD files...")
    uploaded = files.upload()
    paths = []
    for name, data in uploaded.items():
        path = f"/content/{name}"
        with open(path, "wb") as f:
            f.write(data)
        paths.append(path)
    return paths

def build_index_from_paths(paths):
    docs = load_texts_from_paths(paths)
    chunks = chunk_docs(docs)
    rag.build(chunks)
    print(f"Indexed {len(chunks)} chunks from {len(docs)} files.")

In [14]:
# 1) Pick your PDFs / TXT / MD from your computer
paths = upload_files()                 # opens a file picker

# 2) Build the index (chunk + embed + FAISS)
build_index_from_paths(paths)

# 3) Sanity checks
print("Files:", paths)
print("Chunks indexed:", len(rag.chunks))
print("First chunk preview:\n", rag.chunks[0]["text"][:500] if rag.chunks else "No chunks")

Select PDFs/TXT/MD files...


Saving Impact_Agent_Dataset.pdf to Impact_Agent_Dataset.pdf
✓ Loaded PDF: Impact_Agent_Dataset.pdf (17154 chars)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Built index with 26 chunks.
Indexed 26 chunks from 1 files.
Files: ['/content/Impact_Agent_Dataset.pdf']
Chunks indexed: 26
First chunk preview:
 Suicide Related Resources{  1. 988 Suicide & Crisis Lifeline (U.S.) • What it is: A 24/7, free, confidential crisis support line for anyone in emotional distress or suicidal crisis. • How to access: Call or text 988 or chat online at 988lifeline.org. • Why it’s useful: Trained counselors provide immediate emotional support and help connect you to local resources.   2. American Foundation for Suicide Prevention (AFSP) • What it offers: Information on suicide prevention, support for those affected


GRADIO CHAT OVERLAY

In [15]:
import os
import gradio as gr
from pypdf import PdfReader
from docx import Document

def read_file_to_string(file_path: str) -> str:
    """
    Reads a file (TXT, PDF, DOCX, DOC) and returns its contents as a string.
    """
    if not file_path:
        return ""

    text = ""
    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        # PDF files
        if file_ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text += page.extract_text() or ""
            print(f"✓ Loaded PDF: {len(text)} characters")

        # DOCX files
        elif file_ext == ".docx":
            doc = Document(file_path)
            paragraphs = [p.text for p in doc.paragraphs]
            text = "\n".join(paragraphs)
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        text += "\n" + cell.text
            print(f"✓ Loaded DOCX: {len(text)} characters")

        # DOC files (legacy Word)
        elif file_ext == ".doc":
            try:
                doc = Document(file_path)
                paragraphs = [p.text for p in doc.paragraphs]
                text = "\n".join(paragraphs)
                print(f"✓ Loaded DOC: {len(text)} characters")
            except:
                return "⚠️ Error: Legacy .doc format detected. Please save as .docx and try again."

        # Text files
        elif file_ext in [".txt", ".md"]:
            with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                text = f.read()
            print(f"✓ Loaded text file: {len(text)} characters")

        else:
            return f"⚠️ Unsupported file type: {file_ext}"

    except Exception as e:
        return f"⚠️ Error reading file: {str(e)}"

    return text


def _gradio_story_eval(story_text: str, mission: str = ""):
    """
    Runs an Impact-Studios-style evaluation, but grounded in your RAG index
    when available.
    """
    if not story_text.strip():
        return "⚠️ Please enter a valid story, concept, or idea."

    base_prompt = """
    You are an editorial advisor focused on community engagement and ethical publication. Read the following piece of writing
    and identify the communities, audiences, or individuals who may be directly affected by its themes.Then provide specific,
    actionable suggestions for how the author can responsibly and meaningfully reach out to or
    support those communities through or alongside the text.

    Your feedback must:
    -Be concrete (e.g., exact additions, placements, wording ideas, or resources to include).
    -Explain why each suggestion is appropriate for the content.
    -Address ethical considerations such as harm reduction, accessibility, and care for vulnerable readers when relevant.

    Avoid generic advice like "be sensitive" or "raise awareness."

    When recommending a resource, make sure to provide a way to access it such as a phone number or website.

    Do not suggest any changes to the origional content like "consider altering the ending" or "elaborate on this".

    When users share a script, concept, or idea, your job is to:
    1. Analyze the submission's tone, themes, and emotional depth.
    2. Determine the topics that are discussed in the submission.
    3. Provide clear reasoning for your evaluation.
    4. Provide specific, actionable suggestions for how the author can responsibly and meaningfully reach out to or support those communities through or alongside the text.
    5. If any information is provided, integrate it naturally into your evaluation.

    Respond in this format:

    Topics identified:
    (List of topics identified in the submission. Each topic should be listed with at least one quote from the submission that exemplifies the listed topic and at least one specific, actionable recommendations for outreach and support actions)
    """

    if mission and mission.strip():
        user_prompt = f"{base_prompt}\n\nCurrent Studio Mission:\n{mission}\n\nUser Submission:\n{story_text}"
    else:
        user_prompt = f"{base_prompt}\n\nUser Submission:\n{story_text}"

    print("Using backend:", GEN_BACKEND)

    context_text = ""
    if "rag" in globals() and getattr(rag, "chunks", []):
        hits = rag.search(story_text, k=TOP_K)
        context_text = render_context(hits)
        print(f"Injected {len(hits)} RAG chunks as grounding context.")
    else:
        print("No RAG index loaded — using prompt only.")

    full_prompt = (
        ANSWER_TEMPLATE.format(
            system=SYSTEM_PROMPT,
            context=context_text or "[No extra context]",
            question=user_prompt,
        )
    )

    out = answer_question(full_prompt)
    answer = out["answer"]

    cites = []
    if context_text:
        for rank, (_, ch) in enumerate(out["top_chunks"], start=1):
            cites.append(f"Chunk {rank} — {os.path.basename(ch['source'])}#{ch['chunk_id']}")
    suffix = ("\n\n---\nSources: " + "; ".join(cites)) if cites else ""

    return (answer or "[Empty answer]") + suffix


def handle_evaluation(story_text: str, chat_history):
    """
    Handles the evaluation and updates chat history.
    """
    if not story_text.strip():
        return chat_history, ""

    evaluation = _gradio_story_eval(story_text)

    chat_history = chat_history or []
    chat_history.append((story_text, evaluation))

    return chat_history, ""


CUSTOM_CSS = """
/* ===== GLOBAL RESET & FULL WIDTH ===== */
.gradio-container, .contain, .block {
    padding: 0 !important;
    margin: 0 !important;
    max-width: 100% !important;
    width: 100% !important;
}

.gradio-container {
    background: linear-gradient(135deg, #E0F2F7 0%, #B3E5FC 100%) !important;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', sans-serif !important;
    height: 100vh !important;
    display: flex !important;
    flex-direction: column !important;
    overflow: hidden !important;
}

.block {
    border: none !important;
    box-shadow: none !important;
}

/* ===== MAIN LAYOUT ===== */
#main-layout {
    height: 100vh !important;
    display: flex !important;
    flex-direction: row !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    width: 100% !important;
}

/* ===== SIDEBAR ===== */
.sidebar {
    background: #E8F5F9 !important;
    height: 100vh !important;
    padding: 20px !important;
    border-right: 1px solid #B3E5FC !important;
    overflow-y: auto !important;
    flex-shrink: 0 !important;
}

.logo {
    font-size: 16px;
    font-weight: 600;
    color: #01579B;
    background: white;
    padding: 8px 12px;
    border-radius: 6px;
    margin-bottom: 20px;
    display: inline-block;
}

.sidebar-button {
    background: white !important;
    color: #0277BD !important;
    border: 1px solid #81D4FA !important;
    padding: 10px 16px !important;
    margin-bottom: 8px !important;
    border-radius: 8px !important;
    font-size: 14px !important;
    text-align: left !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
    width: 100% !important;
}

.sidebar-button:hover {
    background: #E1F5FE !important;
    border-color: #4FC3F7 !important;
}

.sidebar-section {
    margin-top: 24px;
    padding-top: 16px;
    border-top: 1px solid #B3E5FC;
}

.sidebar-section-title {
    font-size: 12px;
    font-weight: 600;
    color: #0277BD;
    margin-bottom: 8px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

/* ===== MAIN CONTENT AREA ===== */
.main-content {
    background: white !important;
    height: 100vh !important;
    display: flex !important;
    flex-direction: column !important;
    flex: 1 !important;
    min-width: 0 !important;
    overflow: hidden !important;
    width: 100% !important;
    position: relative !important;
}

.header {
    background: white !important;
    border-bottom: 1px solid #E0E0E0 !important;
    padding: 16px 24px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: space-between !important;
    flex-shrink: 0 !important;
    z-index: 10 !important;
}

.header-title {
    font-size: 24px;
    font-weight: 600;
    color: #01579B;
    margin: 0;
}

.share-button {
    background: white !important;
    border: 1px solid #B3E5FC !important;
    color: #0277BD !important;
    padding: 8px 16px !important;
    border-radius: 6px !important;
    font-size: 14px !important;
    cursor: pointer !important;
}

/* ===== CHAT AREA ===== */
.chat-area {
    flex: 1 1 auto !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    background: #E1F5FE !important;
    padding: 24px !important;
    padding-bottom: 100px !important;
    min-height: 0 !important;
    display: flex !important;
    flex-direction: column !important;
}

.chatbot, .chatbot *, .chatbot .wrap, .chatbot .wrap-inner,
.chatbot .panel, .chatbot .panel-inner, .chatbot .scrollable,
.chatbot .scrollable-inner, .chatbot .message-wrap,
.chatbot .message-body, .chatbot .message-container {
    background: transparent !important;
    background-color: transparent !important;
    border: none !important;
    box-shadow: none !important;
    height: auto !important;
}

.chatbot .user-message, .chatbot .bot-message,
.chatbot .message.user, .chatbot .message.bot,
.chatbot [class*="user"], .chatbot [class*="bot"] {
    background: white !important;
    border: 1px solid #B3E5FC !important;
    border-radius: 12px !important;
    padding: 12px 16px !important;
    margin-bottom: 12px !important;
}

/* ===== INPUT BAR - FIXED AT BOTTOM ===== */
.input-container {
    background: white !important;
    border-top: 1px solid #E0E0E0 !important;
    padding: 16px 24px !important;
    position: absolute !important;
    bottom: 0 !important;
    left: 0 !important;
    right: 0 !important;
    z-index: 1000 !important;
    width: 100% !important;
    box-sizing: border-box !important;
}

.input-row {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
    background: white !important;
    border: 1px solid #E0E0E0 !important;
    border-radius: 24px !important;
    padding: 8px 16px !important;
    width: 100% !important;
    box-sizing: border-box !important;
}

/* File Upload - LEFT SIDE - CLEARLY VISIBLE */
.file-upload {
    flex-shrink: 0 !important;
    width: 40px !important;
    min-width: 40px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    position: relative !important;
}

.file-upload .wrap {
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
    margin: 0 !important;
    box-shadow: none !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 40px !important;
    height: 40px !important;
}

.file-upload label {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    font-size: 24px !important;
    color: #0277BD !important;
    cursor: pointer !important;
    padding: 0 !important;
    margin: 0 !important;
    width: 40px !important;
    height: 40px !important;
    background: transparent !important;
    border: none !important;
    line-height: 1 !important;
    visibility: visible !important;
    opacity: 1 !important;
}

.file-upload button {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
    margin: 0 !important;
    width: 40px !important;
    height: 40px !important;
    min-width: 40px !important;
    font-size: 24px !important;
    color: #0277BD !important;
    cursor: pointer !important;
    visibility: visible !important;
    opacity: 1 !important;
}

/* Text Input - CENTER - EXPANDS */
.text-input {
    flex: 1 1 auto !important;
    min-width: 0 !important;
    display: flex !important;
    align-items: center !important;
}

.text-input textarea {
    background: transparent !important;
    border: none !important;
    outline: none !important;
    resize: none !important;
    font-size: 15px !important;
    color: #212121 !important;
    padding: 8px 0 !important;
    margin: 0 !important;
    line-height: 1.4 !important;
    min-height: 24px !important;
    max-height: 60px !important;
    overflow-y: auto !important;
    width: 100% !important;
    font-family: inherit !important;
}

.text-input textarea::placeholder {
    color: #9E9E9E !important;
    opacity: 1 !important;
}

.text-input .wrap {
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
    box-shadow: none !important;
    margin: 0 !important;
    width: 100% !important;
}

.text-input label {
    display: none !important;
}

/* Submit Button - RIGHT SIDE */
.submit-button {
    flex-shrink: 0 !important;
    width: 40px !important;
    min-width: 40px !important;
}

.submit-button button {
    background: #01579B !important;
    color: white !important;
    border: none !important;
    border-radius: 50% !important;
    width: 40px !important;
    height: 40px !important;
    min-width: 40px !important;
    padding: 0 !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    cursor: pointer !important;
    font-size: 20px !important;
    font-weight: bold !important;
    box-shadow: none !important;
}

.submit-button button:hover {
    background: #0277BD !important;
}

label:not(.file-upload label) {
    display: none !important;
}

.input-row > * {
    margin: 0 !important;
}

/* ===== SCROLLBAR ===== */
.chat-area::-webkit-scrollbar {
    width: 6px;
}

.chat-area::-webkit-scrollbar-track {
    background: transparent;
}

.chat-area::-webkit-scrollbar-thumb {
    background: #B3E5FC;
    border-radius: 3px;
}

.chat-area::-webkit-scrollbar-thumb:hover {
    background: #81D4FA;
}

.chatbot, .chatbot .wrap {
    overflow: visible !important;
}

/* ===== MOBILE RESPONSIVE ===== */
@media (max-width: 768px) {
    .sidebar {
        min-width: 200px !important;
    }

    .header {
        padding: 12px 16px !important;
    }

    .chat-area {
        padding: 16px !important;
    }

    .input-container {
        padding: 12px 16px !important;
    }
}
"""

# === Gradio UI ===
with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:

    with gr.Row(elem_id="main-layout"):
        # Left Sidebar
        with gr.Column(scale=0, min_width=240, elem_classes="sidebar"):
            gr.HTML("""
                <div class="sidebar-header">
                    <div class="logo">Impact Studios</div>
                </div>
            """)

            new_chat_btn = gr.Button("📝 New chat", elem_classes="sidebar-button")
            search_btn = gr.Button("🔍 Search chat", elem_classes="sidebar-button")

            gr.HTML("""
                <div class="sidebar-section">
                    <div class="sidebar-section-title">AGENTS/GEMS</div>
                </div>
            """)

            add_agent_btn = gr.Button("➕ Add agent", elem_classes="sidebar-button")

            gr.HTML("""
                <div class="sidebar-section">
                    <div class="sidebar-section-title">YOUR CHATS</div>
                </div>
            """)

        # Main Content Area - vertical column filling viewport
        with gr.Column(scale=1, elem_classes="main-content"):
            # Header - fixed at top
            gr.HTML("""
                <div class="header">
                    <h1 class="header-title">Impact Studios</h1>
                    <button class="share-button">↗ Share</button>
                </div>
            """)

            # Chat Display Area - expands to fill available space
            with gr.Column(scale=1, elem_classes="chat-area"):
                chatbot = gr.Chatbot(
                    label="",
                    show_label=False,
                    container=False,
                    bubble_full_width=False,
                    height=None
                )

            # Input Container - fixed at bottom
            with gr.Column(elem_classes="input-container"):
                with gr.Row(elem_classes="input-row", equal_height=True):
                    # File upload - LEFT - clearly visible with emoji label
                    uploaded_file = gr.File(
                        label="📎",
                        file_types=[".txt", ".pdf", ".docx", ".doc"],
                        type="filepath",
                        show_label=True,
                        container=False,
                        elem_classes="file-upload",
                        scale=0
                    )

                    # Text input - CENTER (expands)
                    story = gr.Textbox(
                        label="",
                        placeholder="Ask anything",
                        lines=1,
                        max_lines=3,
                        show_label=False,
                        container=False,
                        elem_classes="text-input",
                        scale=1
                    )

                    # Submit button (circular arrow) - RIGHT
                    submit_btn = gr.Button(
                        "↑",
                        elem_classes="submit-button",
                        scale=0
                    )

    # Event handlers
    uploaded_file.change(
        fn=read_file_to_string,
        inputs=uploaded_file,
        outputs=story,
    )

    submit_btn.click(
        fn=handle_evaluation,
        inputs=[story, chatbot],
        outputs=[chatbot, story]
    )

    story.submit(
        fn=handle_evaluation,
        inputs=[story, chatbot],
        outputs=[chatbot, story]
    )

    # New chat functionality
    def clear_chat():
        return [], ""

    new_chat_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot, story]
    )

print("Launching Impact Studios Evaluator…")
demo.launch(share=True, debug=True)

/tmp/ipython-input-704589158.py:521: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:
/tmp/ipython-input-704589158.py:521: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:
/tmp/ipython-input-704589158.py:561: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-704589158.py:561: DeprecationWarning: The 'bubble_full_width' p

Launching Impact Studios Evaluator…
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://69aa52b38ae6d4f43c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://69aa52b38ae6d4f43c.gradio.live


In [ ]:
#Use this if first code block fails
import os, gradio as gr

def _gradio_ask(q: str):
    if not q.strip():
        return "Please enter a question."
    print("Using backend:", GEN_BACKEND)  # shows in Colab logs
    out = answer_question(q)
    cites = []
    for rank, (_, ch) in enumerate(out["top_chunks"], start=1):
        cites.append(f"Chunk {rank} — {os.path.basename(ch['source'])}#{ch['chunk_id']}")
    suffix = ("\n\n---\nSources: " + "; ".join(cites)) if cites else ""
    return (out["answer"] or "[Empty answer]") + suffix

with gr.Blocks() as demo:
    gr.Markdown("### RAG Chat — ask questions grounded in your uploaded docs")
    q = gr.Textbox(label="Your question")
    a = gr.Markdown(label="Answer")
    btn = gr.Button("Ask")
    btn.click(_gradio_ask, inputs=q, outputs=a)

print("Launching RAG chat…")
demo.launch(share=True, debug=True)